# LexFrame quickstart

LexFrame reads a hidden state the way the model reads it, in the model's own vocabulary: what the state carries by itself, what the context adds after each layer and from which positions, and how the model uses the state. This notebook reads SmolLM3-3B, compares LexFrame with the logit lens and the Jacobian lens, and steers a prediction. Choose a GPU runtime (Runtime > Change runtime type > T4 GPU).

[Paper](https://arxiv.org/abs/2610.22169) | [Blog post](https://jianguda.github.io/blogs/lexframe/) | [Code](https://github.com/jianguda/lexframe)

In [ ]:
!pip install -q lexframe datasets matplotlib

## Load a model and calibrate a frame

Any causal language model in Hugging Face Transformers works. The frame is calibrated once on unlabelled text, 1024 blocks spread over WikiText-103, which takes a few minutes and is cached on disk.

In [ ]:
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

import lexframe as lf

name = "HuggingFaceTB/SmolLM3-3B"
model = AutoModelForCausalLM.from_pretrained(name, dtype="auto", device_map="cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained(name)
texts = load_dataset("Salesforce/wikitext", "wikitext-103-raw-v1", split="train")["text"]

frame = lf.LexFrame(model, tokenizer).init(texts)
frame

## Read, trace and steer

`state` gives the hidden states of every layer at every position; decoded without context, they show what each state carries by itself. The reading at the last position shows where the answer appears; `trail` traces where the prediction comes from: what the context added after each layer and from which positions, and which layers wrote it; `steer` moves a state along the directions of two tokens.

In [ ]:
text = "The Eiffel Tower is located in the city of"
s = frame.state(text)
labels, probs = s[:, -1].decode(context=False).top(5)
for layer, row in enumerate(labels):
    print(layer, row)

In [ ]:
import numpy as np

tops = labels[:, 0] == " Paris"
print("Paris tops the reading from layer", int(np.argmax(tops)) if tops.any() else -1)
t = frame.trail(text, label=" Paris")
print("the reading of Paris per layer:", np.exp(t.logprobs).round(3))
print("the layers that wrote the prediction most:", t.attribute()[:, 0].argsort()[::-1][:3])
up = t.upstream()  # what each layer's attention brought in from each position; None where it cannot be split
added = np.diff(t.logprobs) if up is None else up.sum(1)
print("the layers whose attention added most:", added.argsort()[::-1][:3])

In [ ]:
c = s[16]  # every position at layer 16
moved = c.steer({" Paris": -4, " London": 4})
print("before:", c.decode().top(5)[0][-1])
print("after: ", moved.decode().top(5)[0][-1])

## Compare with two baselines

The baselines live apart from the frame, in `lexframe.baselines`. The logit lens reads every layer with the output head; the Jacobian lens fits one linear map per layer. Both read every state of a layer alike, whatever its context. Fitting the Jacobian lens takes one backward pass per 8 hidden dimensions and sequence, which adds up on a free GPU: four sequences take several minutes here, and more sequences fit the lens better.

In [ ]:
from lexframe.baselines import JacobianLens, LogitLens

lenses = {"logit lens": LogitLens(model, tokenizer), "Jacobian lens": JacobianLens(model, tokenizer).fit(texts[:1000], size=4)}

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

words = [tokenizer.decode([i]) for i in tokenizer.encode(text, add_special_tokens=False)]
fig, axes = plt.subplots(1, 3, figsize=(18, 8), sharey=True, layout="constrained")
for ax, (title, method) in zip(axes, {**lenses, "LexFrame": frame}.items()):
    r = method.state(text).decode(context=False) if method is frame else method.state(text).decode()
    tokens, probs = r.top(1)
    ax.imshow(np.nan_to_num(probs[..., 0]), aspect="auto", origin="lower", cmap="Blues", vmin=0, vmax=1)
    for (layer, position), token in np.ndenumerate(tokens[..., 0]):
        ax.text(position, layer, repr(token)[1:-1][:10], ha="center", va="center", fontsize=5)
    ax.set_xticks(range(len(words)), words, rotation=45, ha="right")
    ax.set_title(title)
axes[0].set_ylabel("layer (0 is the embedding, the last is the output)")
plt.show()

## More

[docs/USAGE.md](https://github.com/jianguda/lexframe/blob/main/docs/USAGE.md) lists every command, [examples/walkthrough.py](https://github.com/jianguda/lexframe/blob/main/examples/walkthrough.py) goes through them, [examples/baselines.py](https://github.com/jianguda/lexframe/blob/main/examples/baselines.py) plots how far each method's reading lies from the model's output, and [examples/visualize.py](https://github.com/jianguda/lexframe/blob/main/examples/visualize.py) compares their gradients against the ceiling that no fixed lens can pass.